In [13]:
import pandas as pd
import os
import numpy as np
from statsmodels.tsa.ardl import ARDL

# ============================================================
# 1. Load dữ liệu đã xử lý
# ============================================================
data_file = "../data/processed/data_processed.csv"
data_processed = pd.read_csv(data_file)

# ============================================================
# 2. Chọn model EX và IM (có cointegration)
# ============================================================
models = {
    "Model_2_EX": {
        "dep_var": "ln_EX",
        "exog_vars": ["ln_RER_pos","ln_RER_neg","ln_FDI","IPI_VN","IPI_World_GR_diff","ln_M2_diff","ln_WTI","COVID"],
        "lag_dep": 6,
        "lag_exog": [3,1,6,6,5,2,0]
    },
    "Model_3_IM": {
        "dep_var": "ln_IM",
        "exog_vars": ["ln_RER_pos","ln_RER_neg","ln_FDI","IPI_VN","IPI_World_GR_diff","ln_M2_diff","ln_WTI","COVID"],
        "lag_dep": 3,
        "lag_exog": [3,1,2,2,4,3,0]
    }
}


# ============================================================
# 3. Tạo folder results
# ============================================================
output_folder = "../results"

# ============================================================
# 4. Hàm tạo lag thủ công
# ============================================================
def create_lags(df, var_list, lag_list):
    df_lagged = df.copy()
    for var, lag in zip(var_list, lag_list):
        if lag > 0:
            for l in range(1, lag+1):
                df_lagged[f"{var}_L{l}"] = df_lagged[var].shift(l)
    return df_lagged

# ============================================================
# 5. Chạy NARDL estimation
# ============================================================
for model_name, spec in models.items():
    dep = spec["dep_var"]
    exogs = [v for v in spec["exog_vars"] if v in data_processed.columns]

    # Chuẩn bị dataframe và tạo lag
    df_model = data_processed[[dep] + exogs].dropna()
    df_model = create_lags(df_model, exogs, spec["lag_exog"])
    df_model = df_model.dropna()

    # Lấy các exog đã lag
    exog_lagged_vars = [col for col in df_model.columns if col != dep]

    # Fit ARDL/NARDL
    model = ARDL(df_model[dep], lags=spec["lag_dep"], exog=df_model[exog_lagged_vars], trend='c')
    res = model.fit()

    # ============================================================
    # Short-run coefficients
    # ============================================================
    short_run = res.params

    # ============================================================
    # ECM(-1) coefficient
    # ============================================================
    # Lấy hệ số của lag 1 dependent variable
    ecm_coef = res.params.get(f"{dep}.L1", None)

    # ============================================================
    # Long-run coefficients
    # ============================================================
    long_run = {}
    if ecm_coef is not None:
        for orig_var in exogs:
            # Tìm tất cả các lag của biến này trong ARDLResults
            matched_cols = [col for col in res.params.index if col.startswith(orig_var)]
            if matched_cols:
                coef_sum = res.params[matched_cols].sum()
                long_run[orig_var] = -coef_sum / ecm_coef
    else:
        long_run = None

    # ============================================================
    # Wald test bất đối xứng ln_RER_pos vs ln_RER_neg
    # ============================================================
    pos_cols = [col for col in res.params.index if 'ln_RER_pos' in col]
    neg_cols = [col for col in res.params.index if 'ln_RER_neg' in col]
    if pos_cols and neg_cols:
        wald_expr = " + ".join(pos_cols) + " - " + " + ".join(neg_cols) + " = 0"
        wald_res = res.wald_test(wald_expr)
        wald_stat = wald_res.statistic[0][0]
        wald_pval = wald_res.pvalue
    else:
        wald_stat = None
        wald_pval = None

    # ============================================================
    # In kết quả
    # ============================================================
    print("\n" + "="*80)
    print(f"NARDL Estimation for {model_name}")
    print("="*80)
    print("Short-run coefficients:")
    print(short_run)
    print("Long-run coefficients:")
    print(long_run)
    print(f"ECM(-1) coefficient: {ecm_coef}")
    print(f"Wald test statistic: {wald_stat}, p-value: {wald_pval}")

    # ============================================================
    # Xuất CSV
    # ============================================================
    short_run.to_csv(os.path.join(output_folder, f"{model_name}_short_run.csv"))
    if long_run is not None:
        pd.Series(long_run).to_csv(os.path.join(output_folder, f"{model_name}_long_run.csv"))
    pd.DataFrame({
        'ECM(-1)': [ecm_coef],
        'Wald_statistic': [wald_stat],
        'Wald_p_value': [wald_pval]
    }).to_csv(os.path.join(output_folder, f"{model_name}_ecm_wald.csv"))


NARDL Estimation for Model_2_EX
Short-run coefficients:
const                2.737567
ln_EX.L1             0.336600
ln_EX.L2             0.268898
ln_EX.L3             0.202475
ln_EX.L4             0.014805
ln_EX.L5             0.171483
ln_EX.L6            -0.185016
ln_RER_pos.L0       -2.697790
ln_RER_neg.L0        3.084184
ln_FDI.L0            0.153525
IPI_VN.L0            0.007020
ln_M2_diff.L0      -10.823339
ln_WTI.L0            0.168369
ln_RER_pos_L1.L0    -1.175238
ln_RER_pos_L2.L0    -4.897027
ln_RER_pos_L3.L0     4.180073
ln_RER_neg_L1.L0    -4.545936
ln_FDI_L1.L0         0.306876
ln_FDI_L2.L0        -0.501357
ln_FDI_L3.L0         0.378352
ln_FDI_L4.L0        -0.230219
ln_FDI_L5.L0        -0.139638
ln_FDI_L6.L0         0.181550
IPI_VN_L1.L0        -0.003519
IPI_VN_L2.L0        -0.001869
IPI_VN_L3.L0        -0.002381
IPI_VN_L4.L0         0.000470
IPI_VN_L5.L0        -0.002650
IPI_VN_L6.L0         0.004707
ln_M2_diff_L1.L0   -35.853200
ln_M2_diff_L2.L0   -18.580075
ln_M2_diff_L3

c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\

In [10]:
import pandas as pd
import os

# ============================================================
# 1. Thư mục chứa các file CSV hiện tại
# ============================================================
input_folder = "../results"  # thư mục chứa các file CSV của NARDL
output_file = os.path.join(input_folder, "NARDL_combined_results.csv")

# ============================================================
# 2. Danh sách các file theo model
# ============================================================
files = {
    "Model_2_EX": {
        "short_run": "Model_2_EX_short_run.csv",
        "long_run": "Model_2_EX_long_run.csv",
        "ecm_wald": "Model_2_EX_ecm_wald.csv",
        "dep": "ln_EX"
    },
    "Model_3_IM": {
        "short_run": "Model_3_IM_short_run.csv",
        "long_run": "Model_3_IM_long_run.csv",
        "ecm_wald": "Model_3_IM_ecm_wald.csv",
        "dep": "ln_IM"
    }
}

# ============================================================
# 3. Gộp tất cả vào 1 DataFrame
# ============================================================
combined_list = []

for model_name, paths in files.items():
    # Load các file
    short_run = pd.read_csv(os.path.join(input_folder, paths["short_run"]), index_col=0)
    long_run = pd.read_csv(os.path.join(input_folder, paths["long_run"]), index_col=0)
    ecm_wald = pd.read_csv(os.path.join(input_folder, paths["ecm_wald"]))  # index=False khi xuất

    # Chuẩn bị row tổng hợp
    row = {}
    row["Model"] = model_name
    row["Dependent"] = paths["dep"]

    # Short-run coefficients
    for col, val in short_run.iloc[:,0].items():
        row[f"SR_{col}"] = val

    # Long-run coefficients
    if not long_run.empty:
        for col, val in long_run.iloc[:,0].items():
            row[f"LR_{col}"] = val

    # ECM(-1) + Wald test (dòng đầu tiên)
    row["ECM(-1)"] = ecm_wald.loc[0, "ECM(-1)"]
    row["Wald_stat"] = ecm_wald.loc[0, "Wald_statistic"]
    row["Wald_pval"] = ecm_wald.loc[0, "Wald_p_value"]

    combined_list.append(row)

# ============================================================
# 4. Tạo DataFrame cuối cùng
# ============================================================
df_combined = pd.DataFrame(combined_list)

# Xuất CSV
df_combined.to_csv(output_file, index=False)
print(f"Combined NARDL results exported to: {output_file}")

Combined NARDL results exported to: ../results\NARDL_combined_results.csv
